# Comparación de los modelos

Este notebook no vuelve a entrenar. Lee lo que `01_experimentos.ipynb` guardó en MLflow y compara esos entrenamientos.

Cada entrenamiento es un run del experimento `/Users/<usuario>/bigdata_emanuel_rb_tp_final`. El run tiene un nombre, los parámetros de esa corrida y las métricas de train y de test. Acá entran solo los que terminaron. Si el mismo nombre se entrenó más de una vez, queda el más reciente.

La comparación fija son cuatro runs: `exp_01`, `exp_02`, `exp_03` y `exp_04`. Un entrenamiento en modo manual también aparece, con el nombre que se le haya puesto. El elegido es el de mayor F1 macro de test.

## La tabla de runs

La celda busca los runs en MLflow y arma una fila por nombre. Las columnas son el nombre del run, el modelo, el modo de entrenamiento, el tratamiento del BPM, el balanceo, `C`, la cantidad de árboles, la profundidad, los puntajes de salud, el F1 macro de train, el F1 macro de test, la accuracy de test y el F1 de cada clase.

Esas métricas las escribió `01_experimentos.ipynb` al entrenar. `mlflow.autolog()` guardó el modelo en ese momento. Acá no se vuelve a cargar: la tabla se ordena por el F1 macro de test, de mayor a menor. La accuracy está para compararla, y no elige al ganador.

In [ ]:
import logging
import mlflow
import pandas as pd
import matplotlib.pyplot as plt

logging.getLogger("pyspark.sql.connect").setLevel(logging.ERROR)
usuario = spark.sql("SELECT current_user()").first()[0]
experimento = f"/Users/{usuario}/bigdata_emanuel_rb_tp_final"
exp = mlflow.get_experiment_by_name(experimento)
if exp is None:
    raise RuntimeError(f"No existe el experimento {experimento}. Primero hay que correr 01_experimentos.ipynb.")

runs = mlflow.search_runs([exp.experiment_id])
if runs.empty:
    raise RuntimeError(f"El experimento {experimento} no tiene runs. Primero hay que correr 01_experimentos.ipynb.")
runs = runs[runs["status"] == "FINISHED"].copy()
if runs.empty:
    raise RuntimeError(f"El experimento {experimento} no tiene runs terminados.")
if "tags.mlflow.runName" in runs.columns:
    runs["run"] = runs["tags.mlflow.runName"]
else:
    runs["run"] = runs["run_name"]
runs = runs.sort_values("start_time", ascending=False).drop_duplicates("run")

columnas = {
    "run": "run",
    "params.modelo": "modelo",
    "params.que_entrenar": "que_entrenar",
    "params.tratar_bpm": "tratar_bpm",
    "params.balanceo": "balanceo",
    "params.C": "C",
    "params.n_arboles": "n_arboles",
    "params.profundidad": "profundidad",
    "params.puntajes_salud": "puntajes_salud",
    "metrics.train_f1_macro": "train_f1_macro",
    "metrics.test_f1_macro": "test_f1_macro",
    "metrics.test_accuracy": "test_accuracy",
    "metrics.test_f1_improve": "f1_improve",
    "metrics.test_f1_no_effect": "f1_no_effect",
    "metrics.test_f1_worsen": "f1_worsen",
}
for columna in columnas:
    if columna not in runs.columns:
        runs[columna] = None
comparacion = runs.rename(columns=columnas)[list(columnas.values())]
comparacion = comparacion.dropna(subset=["test_f1_macro"])
if comparacion.empty:
    raise RuntimeError("Ningún run tiene F1 macro de test. Hay que volver a entrenar en 01_experimentos.ipynb.")
for metrica in ["train_f1_macro", "test_f1_macro", "test_accuracy", "f1_improve", "f1_no_effect", "f1_worsen"]:
    comparacion[metrica] = comparacion[metrica].astype(float)
comparacion = comparacion.sort_values("test_f1_macro", ascending=False).reset_index(drop=True)
del runs

print(f"Experimento: {experimento}")
print(f"Runs terminados: {len(comparacion)}. Ordenados por F1 macro de test, de mayor a menor.")
display(comparacion.round(4))

nombres = ["exp_01", "exp_02", "exp_03", "exp_04"]
faltan = [nombre for nombre in nombres if nombre not in set(comparacion["run"])]
if faltan:
    print(f"Faltan runs de la comparación fija: {faltan}. Hay que correr 01_experimentos.ipynb con Como entrenar en automatico.")

## Qué run queda elegido

Gana el run con mayor F1 macro de test. El criterio está fijado antes de mirar la tabla: las tres clases pesan igual, así que el promedio del F1 de `Improve`, `No effect` y `Worsen` decide.

La segunda fila muestra quién iría primero si se mirara la accuracy. `Improve` es cerca del 74 % de las respuestas. Un modelo que casi siempre elige esa clase acierta muchas filas, aunque falle en las otras dos. Por eso el primero por accuracy puede no ser el elegido.

In [ ]:
ver = ["run", "modelo", "que_entrenar", "tratar_bpm", "balanceo", "puntajes_salud", "C", "n_arboles", "profundidad", "train_f1_macro", "test_f1_macro", "test_accuracy"]
por_f1 = comparacion.sort_values("test_f1_macro", ascending=False).head(1)
por_accuracy = comparacion.sort_values("test_accuracy", ascending=False).head(1)
display(pd.concat([
    por_f1.assign(criterio="mejor F1 macro"),
    por_accuracy.assign(criterio="mejor accuracy"),
])[["criterio"] + ver].round(4))
elegido = por_f1.iloc[0]
print(
    f"Elegido por F1 macro de test: {elegido['run']} ({float(elegido['test_f1_macro']):.4f}). "
    f"En train ese F1 es {float(elegido['train_f1_macro']):.4f}."
)
print(
    f"Por accuracy iría primero {por_accuracy.iloc[0]['run']} "
    f"({float(por_accuracy.iloc[0]['test_accuracy']):.4f}). Esa métrica no elige al modelo."
)

## Cómo leer los gráficos

El primer gráfico ordena los mismos runs por F1 macro de test. El color es el modelo: regresión logística en azul, random forest en verde, LightGBM en naranja y CatBoost en violeta.

El segundo pone el F1 macro de train en el eje horizontal y el de test en el vertical. La línea punteada marca donde los dos valores coinciden. Un punto sobre esa línea tuvo un resultado parecido en train y en test. Un punto por debajo y a la derecha acertó mucho mejor el train que el test: memorizó ese corte.

In [ ]:
colores = {
    "logistica": "#2563eb",
    "regresion_logistica": "#2563eb",
    "random_forest": "#16a34a",
    "lightgbm": "#f59e0b",
    "catboost": "#7c3aed",
}
ordenado = comparacion.sort_values("test_f1_macro")
fig, axes = plt.subplots(1, 2, figsize=(12, max(3.5, 0.38 * len(comparacion) + 1.2)))
axes[0].barh(
    ordenado["run"],
    ordenado["test_f1_macro"],
    color=[colores.get(modelo, "#94a3b8") for modelo in ordenado["modelo"]],
)
axes[0].set_xlabel("F1 macro de test")
axes[0].set_title("F1 macro de test por run")
axes[1].scatter(
    comparacion["train_f1_macro"],
    comparacion["test_f1_macro"],
    s=60,
    color=[colores.get(modelo, "#94a3b8") for modelo in comparacion["modelo"]],
)
for fila in comparacion.itertuples():
    axes[1].annotate(fila.run, (fila.train_f1_macro, fila.test_f1_macro), fontsize=7, xytext=(4, 3), textcoords="offset points")
ejes = [0, 1.02]
axes[1].plot(ejes, ejes, color="#94a3b8", ls="--")
axes[1].set_xlim(*ejes)
axes[1].set_ylim(*ejes)
axes[1].set_xlabel("F1 macro de train")
axes[1].set_ylabel("F1 macro de test")
axes[1].set_title("Train contra test")
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Los mismos runs, en texto

Imprime la misma comparación, del mayor al menor F1 macro de test. Cada fila es un run que `01_experimentos.ipynb` registró en MLflow, con los parámetros de esa corrida y sus métricas. No agrega runs ni cambia el orden.

In [ ]:
copiar = ["run", "modelo", "que_entrenar", "tratar_bpm", "balanceo", "puntajes_salud", "train_f1_macro", "test_f1_macro", "test_accuracy", "f1_improve", "f1_no_effect", "f1_worsen"]
vista = comparacion[copiar].round(4).fillna("")
lineas = ["| " + " | ".join(copiar) + " |", "|" + "|".join(["---"] * len(copiar)) + "|"]
for _, fila in vista.iterrows():
    lineas.append("| " + " | ".join(str(fila[col]) for col in copiar) + " |")
print("\n".join(lineas))

## Por qué ese criterio

El modelo de esta comparación es el que nombra la celda **Qué run queda elegido**. Es el mayor F1 macro de test entre los runs que el notebook acaba de leer. El criterio no se cambia después de ver quién quedó primero.

El F1 macro promedia el F1 de las tres clases. La accuracy cuenta filas bien clasificadas, y la mayoría de las filas son `Improve`. Un modelo puede subir la accuracy acertando esa clase y aun así tener un F1 bajo en `No effect` o en `Worsen`. En el test de este corte hay unas 3 filas `Worsen`. Si el F1 de esa clase es 0, ningún modelo las clasificó bien.

Conviene mirar también el F1 de train de esa misma fila. Si está en 1 y el de test queda mucho más abajo, el modelo acertó todas las filas de entrenamiento y no sostuvo ese resultado en el test.
